# Reproducing the Yb171 `total_decay` range-expansion issue

This notebook gives a lightweight code demo for the `rydcalc` behavior discussed in the slides.  It does **not** run the full lifetime calculation.  Instead, it monkey-patches the expensive MQDT `boundstatesinrange` calls with a recorder that immediately returns an empty state list.

That lets us observe the range requests made by `yb.total_decay(...)` itself.  The result proves that, for
$|r\rangle = |65\,{}^3S_1,F=3/2,m_F=-3/2\rangle$, `total_decay` asks the Yb171 MQDT solvers to search roughly
$\nu \in [0.001,129.121]$ before any final-state filtering can help.

Run this notebook with the project Python 3.12 environment.  A system Python 3.9 kernel may fail while importing `rydcalc` dependencies because of NumPy/Matplotlib ABI mismatch.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-rydcalc")

from neutral_yb.external.rydcalc_adapter import build_yb171_atom, load_rydcalc  # noqa: E402

rydcalc = load_rydcalc(require_c_extension=True)
yb = build_yb171_atom(cpp_numerov=True, use_db=False, require_c_extension=True)
initial_state = yb.get_state((65, 0, 1.5, -1.5), tt="vlfm")

expected_lo = initial_state.nub - initial_state.n
expected_hi = initial_state.nub + initial_state.n
nmax = initial_state.n + 10

print("Target state attributes")
print('  input label: (65, 0, 1.5, -1.5), tt="vlfm"')
print(f"  state: {initial_state}")
print(f"  st.qn = {initial_state.qn}")
print(f"  st.n = {initial_state.n!r}")
print(f"  st.nub = {initial_state.nub:.8f}")
print(f"  nmax = st.n + 10 = {nmax:.8f}")
print(
    "  expected get_nearby range = [st.nub - st.n, st.nub + st.n] = "
    f"[{expected_lo:.8f}, {expected_hi:.8f}]"
)

Target state attributes
  input label: (65, 0, 1.5, -1.5), tt="vlfm"
  state: |171Yb:64.56,L=0,F=1.5,-1.5>
  st.qn = (64.56, 1, 1.5, -1.5)
  st.n = 64.56
  st.nub = 64.56106363
  nmax = st.n + 10 = 74.56000000
  expected get_nearby range = [st.nub - st.n, st.nub + st.n] = [0.00106363, 129.12106363]


The next cell records the ranges requested by `total_decay` without doing any real MQDT root search.

Two stubs are installed temporarily:

- every MQDT solver's `boundstatesinrange` is replaced by a recorder returning empty arrays;
- `yb.partial_decay` is replaced by a zero-rate function, because this demo is about basis generation, not the dipole-rate formula.

We also pass `include_fn=lambda s: False` into `total_decay`.  If the public `include_fn` could control the expensive front-end search, this should prevent range requests.  It does not.

In [2]:
def install_range_recorder(atom):
    calls = []
    originals = []
    for item in atom.mqdt_models:
        solver = item["model"]
        original = solver.boundstatesinrange
        originals.append((solver, original))
        label = (item["L"], item["F"])

        def recorder(range_arg, *, label=label):
            lo, hi = float(range_arg[0]), float(range_arg[1])
            calls.append(
                {
                    "L": label[0],
                    "F": label[1],
                    "lo": lo,
                    "hi": hi,
                    "width": hi - lo,
                }
            )
            return [np.array([], dtype=float), np.array([], dtype=float)]

        solver.boundstatesinrange = recorder
    return calls, originals


def restore_boundstatesinrange(originals):
    for solver, original in originals:
        solver.boundstatesinrange = original


original_partial_decay = yb.partial_decay
yb.partial_decay = lambda st, other, env: 0.0
calls, originals = install_range_recorder(yb)
try:
    env = rydcalc.environment(T_K=0)
    gamma = yb.total_decay(initial_state, env, include_fn=lambda s: False)
finally:
    restore_boundstatesinrange(originals)
    yb.partial_decay = original_partial_decay

print("Recorded boundstatesinrange calls from yb.total_decay(...)")
print(f"  gamma returned by stubbed partial_decay = {gamma}")
print(f"  number of MQDT range requests = {len(calls)}")
for index, call in enumerate(calls, start=1):
    print(
        f"  {index}. L={call['L']}, F={call['F']}: "
        f"[{call['lo']:.8f}, {call['hi']:.8f}], width={call['width']:.8f}"
    )

Recorded boundstatesinrange calls from yb.total_decay(...)
  gamma returned by stubbed partial_decay = 0.0
  number of MQDT range requests = 5
  1. L=0, F=0.5: [0.00106363, 129.12106363], width=129.12000000
  2. L=0, F=1.5: [0.00106363, 129.12106363], width=129.12000000
  3. L=1, F=0.5: [0.00106363, 129.12106363], width=129.12000000
  4. L=1, F=1.5: [0.00106363, 129.12106363], width=129.12000000
  5. L=1, F=2.5: [0.00106363, 129.12106363], width=129.12000000


## What this proves

1. `total_decay` passes `dn=st.n` to `single_basis.fill`.
2. For the Yb171 MQDT state, `st.n` is `64.56`, i.e. an effective quantum-number label, not the integer principal quantum number `65`.
3. `Ytterbium171.get_nearby` applies that `dn` directly as a half-width in `nub`, so every relevant MQDT manifold receives the search interval `[0.00106363, 129.12106363]`.
4. The public `include_fn=lambda s: False` passed to `total_decay` does not prevent those range requests, because `total_decay` replaces it with its own internal lambda and because filtering occurs after `get_nearby` returns.
5. This reproduces the core bug from another angle: without running the slow root finder, we can see that the expensive and numerically dangerous range expansion is requested before any final-state filtering.